In [1]:
import os
from nnsight import LanguageModel

os.environ["CUDA_VISIBLE_DEVICES"] = "5"

model = LanguageModel(
    "Qwen/Qwen3-32B",
    attn_implementation="eager"
)

/disk/u/gio/.conda/envs/liminal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model.to_empty(device="cuda:0")
model.device

device(type='cuda', index=0)

In [58]:
import json

data = []
with open('../data/exp_2_hsp_res.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))
        
DATA = data[1]
        
DATA['answer']

10

In [59]:
full_cot = DATA['full_cot']

def extract_cot_minus_answer(full_cot):
    # Calculate the index of the last character of "\\boxed{"
    index_of_boxed = full_cot.rfind("\\boxed{", )
    index_of_answer = index_of_boxed + +len("\\boxed{")
    return full_cot[:index_of_answer]
    
cot_minus_answer = extract_cot_minus_answer(full_cot)

def get_target_token_id(full_cot, cot_minus_answer):
    """Get the id of the first token of the final answer."""
    token_ids = model.tokenizer.encode(full_cot, add_special_tokens=False)
    prefix_tokens = model.tokenizer.encode(cot_minus_answer, add_special_tokens=False)
    target_token_id = token_ids[len(prefix_tokens)]
    return target_token_id
    
target_id = get_target_token_id(full_cot, cot_minus_answer)

In [63]:
print(target_id)
print(model.tokenizer.decode(target_id))

16
1


In [60]:
def extract_cot_minus_answer(full_cot):
    # Calculate the index of the last character of "\\boxed{"
    index_of_boxed = full_cot.rfind("\\boxed{", )
    index_of_answer = index_of_boxed + +len("\\boxed{")
    return full_cot[:index_of_answer]
    
extract_cot_minus_answer(DATA['full_cot'])

'<think>\nOkay, so I need to figure out how many continuous paths there are from point A to point B along the segments of this figure without revisiting any of the six labeled points. Let me start by visualizing the Asymptote figure they provided. \n\nFirst, there\'s a rectangle with vertices at (0,0), (3,0), (3,2), and (0,2). Then there are some additional lines: from (0,2) to (1,0), from (1,0) to (3,2), and from (0,2) to (1.5,3.5), which is point A. Also, there\'s a line from (1.5,3.5) to (3,2). The labeled points are A at the top, B at the bottom-left, C at the top-left, D at the top-right, E at the bottom-right, F at the middle-bottom. \n\nSo, the figure is a combination of a rectangle with a diagonal-like structure inside and a peak at the top. Let me try to sketch this mentally. The main rectangle is 3 units wide and 2 units tall. Then from point C (0,2), there\'s a line going down to F (1,0), then up to D (3,2), and back to C. Also, from C, there\'s a line going up to A (1.5,3.5

In [61]:
model.tokenizer.tokenize(full_cot)[-10:]

['.ĊĊ', '$$', 'Ċ', '\\', 'boxed', '{', '1', '0', '}Ċ', '$$']

In [ ]:
print(cot_minus_answer)

<think>
Okay, so I need to figure out how many continuous paths there are from point A to point B along the segments of this figure without revisiting any of the six labeled points. Let me start by visualizing the Asymptote figure they provided. 

First, there's a rectangle with vertices at (0,0), (3,0), (3,2), and (0,2). Then there are some additional lines: from (0,2) to (1,0), from (1,0) to (3,2), and from (0,2) to (1.5,3.5), which is point A. Also, there's a line from (1.5,3.5) to (3,2). The labeled points are A at the top, B at the bottom-left, C at the top-left, D at the top-right, E at the bottom-right, F at the middle-bottom. 

So, the figure is a combination of a rectangle with a diagonal-like structure inside and a peak at the top. Let me try to sketch this mentally. The main rectangle is 3 units wide and 2 units tall. Then from point C (0,2), there's a line going down to F (1,0), then up to D (3,2), and back to C. Also, from C, there's a line going up to A (1.5,3.5), and fro

In [66]:
print(target_id)
print(model.tokenizer.decode(target_id))

16
1


In [ ]:
import torch
from tqdm import trange

def integrated_gradients(
    model: LanguageModel,
    input_text: str,
    target_token_id: int, # Which output token to attribute
    baseline_id: int | None = None,
    interpolation_steps: int = 50
):
    if baseline_id is None:
        baseline_id = model.tokenizer.pad_token_id or model.tokenizer.eos_token_id
        
    # Get the baseline embedding
    baseline_embed = model.model.embed_tokens.weight[baseline_id].detach()
    
    # Tokenize the full input
    token_ids = model.tokenizer.encode(input_text, add_special_tokens=False)
    tokens = model.tokenizer.tokenize(input_text)
    
    # Get all token embeddings: shape (seq_len, hidden_dim)
    token_embeds = model.model.embed_tokens.weight[token_ids].detach()
    
    # Baseline is repeated for each position: shape (seq_len, hidden_dim)
    baseline_embeds = baseline_embed.unsqueeze(0).expand_as(token_embeds)
    
    # Difference between input and baseline
    x_minus_baseline = token_embeds - baseline_embeds # (seq_len, hidden_dim)
    
    # Accumulate gradients across interpolation steps
    accumulated_grads = torch.zeros_like(token_embeds).to(device="cpu")
    print(accumulated_grads.device)
    
    for step in trange(1, interpolation_steps + 1):
        alpha = step / interpolation_steps
        interpolated_embeds = baseline_embeds + alpha * x_minus_baseline
        
        with model.trace(input_text):
            # Move to correct device and add batch dimension INSIDE trace
            interpolated_embeds_traced = interpolated_embeds.unsqueeze(0).to(model.device).requires_grad_(True)
            
            # Override the embedding output
            model.model.embed_tokens.output = interpolated_embeds_traced
            
            # Get logits
            logits = model.output[0]
            target_logit = logits[0, -1, target_token_id]
            
            # Compute gradients
            target_logit.backward()
            grad = interpolated_embeds_traced.grad.save()
            
        print(f"{grad.device=}")
        
        accumulated_grads += grad.squeeze(0)
        
    # Average the gradients
    avg_grads = accumulated_grads / interpolation_steps
    
    # Integrated gradients = (x - x') * avg_grads
    ig_attributions = x_minus_baseline * avg_grads # (seq_len, hidden_dim)
    
    # sum across hidden dimension to get per-token attributino
    token_attributions = ig_attributions.sum(dim=-1) # (seq_len,)
    
    return {
        'tokens': tokens,
        'token_ids': token_ids,
        "attributions": token_attributions,
        "attributions_full": ig_attributions,
    }
        
result = integrated_gradients(
    model=model,
    input_text=cot_minus_answer,
    target_token_id=target_id,
    interpolation_steps=50
)

for token, attr in zip(result["tokens"], result["attributions"]):
    print(f"{token:>10}: {attr.item():+.4f}")

cpu


  2%|▏         | 1/50 [19:25<15:51:54, 1165.59s/it]

grad.device=device(type='cpu')


  4%|▍         | 2/50 [40:58<16:32:16, 1240.34s/it]

grad.device=device(type='cpu')


...

grad.device=device(type='cpu')


 10%|█         | 5/50 [1:43:10<15:30:55, 1241.23s/it]

grad.device=device(type='cpu')


 12%|█▏        | 6/50 [2:03:13<15:00:51, 1228.44s/it]

grad.device=device(type='cpu')


In [ ]:
def to_json_safe(obj):
    if torch.is_tensor(obj):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_json_safe(x) for x in obj]
    else:
        return obj

with open('../data/attributions/result2.json', 'w') as f:
    json.dump(to_json_safe(result), f, indent=2)

In [56]:
import matplotlib.pyplot as plt
import numpy as np

def plot_token_attributions(result, title="Integrated Gradients Attribution"):
    tokens = [tok.replace('Ġ', ' ') for tok in result["tokens"]]
    attributions = result["attributions"].detach().cpu().numpy()
    
    # Normalize for color mapping
    abs_max = np.abs(attributions).max()
    normalized = attributions / abs_max if abs_max > 0 else attributions
    
    fig, ax = plt.subplots(figsize=(len(tokens) * 1.5, 2))
    
    # Color map: red for positive, blue for negative
    colors = plt.cm.RdBu_r((normalized + 1) / 2)
    
    # Create colored boxes for each token
    for i, (token, attr, color) in enumerate(zip(tokens, attributions, colors)):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=color, edgecolor='black', linewidth=1))
        
        # Token text
        ax.text(i + 0.5, 0.5, token, ha='center', va='center', fontsize=12, fontweight='bold')
        
        # Attribution value below
        ax.text(i + 0.5, -0.2, f"{attr:.3f}", ha='center', va='top', fontsize=9)
    
    ax.set_xlim(0, len(tokens))
    ax.set_ylim(-0.5, 1.2)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14, pad=10)
    
    # Colorbar
    sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(-abs_max, abs_max))
    cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', fraction=0.05, pad=0.3)
    cbar.set_label('Attribution Score')
    
    #plt.tight_layout()
    return fig, ax


# Alternative: horizontal bar chart (better for longer sequences)
def plot_token_attributions_bar(result, title="Integrated Gradients Attribution"):
    tokens = [tok.replace('Ġ', ' ').replace('Ċ', ' ').replace('$', r'\$') for tok in result["tokens"]]
    attributions = result["attributions"].detach().cpu().numpy()
    
    fig, ax = plt.subplots(figsize=(8, len(tokens) * 0.5 + 1))
    
    colors = ['#d73027' if a > 0 else '#4575b4' for a in attributions]
    y_pos = np.arange(len(tokens))
    
    ax.barh(y_pos, attributions, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(tokens, fontsize=11)
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Attribution Score')
    ax.set_title(title)
    
    #plt.tight_layout()
    return fig, ax


# Usage
#fig1, ax1 = plot_token_attributions(result, title=f"Attribution for predicting '{model.tokenizer.decode([target_id])}'")
#plt.show()

fig2, ax2 = plot_token_attributions_bar(result, title=f"Attribution for predicting '{model.tokenizer.decode([target_id])}'")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [42]:
def to_json_safe(obj):
    if torch.is_tensor(obj):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_json_safe(x) for x in obj]
    else:
        return obj

with open('../data/result.json', 'w') as f:
    json.dump(to_json_safe(result), f, indent=2)

In [1]:
import json

#with open('../data/MATH_100/idx0_attribution.json', 'r') as f:
with open('../data/attributions/result1.json', 'r') as f:
    result = json.load(f)

In [2]:
result.keys()

dict_keys(['tokens', 'token_ids', 'attributions', 'attributions_full'])

In [4]:
result['tokens']

['<think>',
 'Ċ',
 'Okay',
 ',',
 'Ġso',
 'ĠI',
 'Ġneed',
 'Ġto',
 'Ġfind',
 'Ġ$',
 'Ġx',
 '^',
 '2',
 'Ġ+',
 'Ġy',
 '^',
 '2',
 'Ġ$',
 'Ġgiven',
 'Ġthat',
 'Ġthe',
 'Ġarithmetic',
 'Ġmean',
 'Ġof',
 'Ġ$',
 'Ġx',
 'Ġ$',
 'Ġand',
 'Ġ$',
 'Ġy',
 'Ġ$',
 'Ġis',
 'Ġ',
 '7',
 'Ġand',
 'Ġtheir',
 'Ġgeometric',
 'Ġmean',
 'Ġis',
 'Ġ$',
 'Ġ\\',
 'sqrt',
 '{',
 '1',
 '9',
 '}',
 'Ġ$.',
 'Let',
 'Ġme',
 'Ġstart',
 'Ġby',
 'Ġrecalling',
 'Ġwhat',
 'Ġarithmetic',
 'Ġand',
 'Ġgeometric',
 'Ġmeans',
 'Ġare',
 '.The',
 'Ġarithmetic',
 'Ġmean',
 'Ġof',
 'Ġtwo',
 'Ġnumbers',
 'Ġ$',
 'Ġx',
 'Ġ$',
 'Ġand',
 'Ġ$',
 'Ġy',
 'Ġ$',
 'Ġis',
 'Ġ$',
 'Ġ\\',
 'frac',
 '{x',
 'Ġ+',
 'Ġy',
 '}{',
 '2',
 '}',
 'Ġ$',
 ',',
 'Ġright',
 '?',
 'And',
 'Ġthe',
 'Ġproblem',
 'Ġsays',
 'Ġthis',
 'Ġis',
 'Ġequal',
 'Ġto',
 'Ġ',
 '7',
 '.So',
 'ĠI',
 'Ġcan',
 'Ġwrite',
 'Ġthat',
 'Ġas',
 'Ġan',
 'Ġequation',
 ':ĊĊ',
 '$$',
 'Ċ',
 '\\',
 'frac',
 '{x',
 'Ġ+',
 'Ġy',
 '}{',
 '2',
 '}',
 'Ġ=',
 'Ġ',
 '7',
 'Ċ',
 '$$',
 'ĊĊ',
 'I

In [ ]:
import numpy as np
from typing import Dict, List

def normalize_segment_strengths(segment_data: List[Dict]):
    total_strength = 0
    for item in segment_data:
        total_strength += item['normalized_strength_pre']
        
    for item in segment_data:
        norm_strength_pre = item['normalized_strength_pre']
        item['normalized_strength_post'] = norm_strength_pre / total_strength
        
    return segment_data
        

def segment_normalized_strengths(result: dict):
    tokens = result["tokens"]
    attributions = result["attributions"]

    pre_delimiters = {'Ġ$.', '?', 'ĊĊ', '!', '...', '.', ').', ').ĊĊ'} # '.'
    post_delimiters = {'.The', '.So', '.Now', '.L', '.First', '.Then', '.There'}

    segment_strengths = []
    start = 0
    for i, token in enumerate(tokens):
        if token in pre_delimiters:
            seg_toks = tokens[start:i+1]
            seg_attrs = attributions[start:i+1]
            strength = float((np.sum(np.asarray([np.abs(attr) for attr in seg_attrs]))))
            normalized_strength_1 = strength / np.sqrt(len(seg_attrs))
            segment_strengths.append(normalized_strength)
            start = i + 1

        elif token in post_delimiters:
            seg_attrs = attributions[start:i]   # delimiter starts new segment
            strength = float((np.sum(np.asarray([np.abs(attr) for attr in seg_attrs]))))
            normalized_strength = strength / np.sqrt(len(seg_attrs))
            segment_strengths.append(normalized_strength)
            start = i
            
    return segment_strengths

def segment_strengths_and_consistencies(result: dict):
    tokens = result["tokens"]
    attributions = result["attributions"]

    pre_delimiters = {'Ġ$.', '?', 'ĊĊ', '!', '...', '.', ').', ').ĊĊ'} # '.'
    post_delimiters = {'.The', '.So', '.Now', '.L', '.First', '.Then', '.There'}

    segment_data = []
    start = 0
    for i, token in enumerate(tokens):
        if token in pre_delimiters:
            seg_toks = tokens[start:i+1]
            seg_attrs = attributions[start:i+1]
            strength = float((np.sum(np.asarray([np.abs(attr) for attr in seg_attrs]))))
            normalized_strength = strength / np.sqrt(len(seg_attrs))
            consistency_numerator = float(np.abs(np.sum(np.asarray(seg_attrs))))
            consistency_denominator = float((np.sum(np.asarray([np.abs(attr) for attr in seg_attrs]))))
            seg_data = {
                'tokens': seg_toks,
                'strength': strength,
                'normalized_strength_pre': normalized_strength,
                'consistency': consistency_numerator / consistency_denominator
            }
            segment_data.append(seg_data)
            start = i + 1
        elif token in post_delimiters:
            seg_toks = tokens[start:i]
            seg_attrs = attributions[start:i]
            strength = float((np.sum(np.asarray([np.abs(attr) for attr in seg_attrs]))))
            normalized_strength = strength / np.sqrt(len(seg_attrs))
            consistency_numerator = float(np.abs(np.sum(np.asarray(seg_attrs))))
            consistency_denominator = float((np.sum(np.asarray([np.abs(attr) for attr in seg_attrs]))))
            seg_data = {
                'tokens': seg_toks,
                'strength': strength,
                'normalized_strength_pre': normalized_strength,
                'consistency': consistency_numerator / consistency_denominator
            }
            segment_data.append(seg_data)
            start = i
    
    return segment_data
            
segment_data_pre = segment_strengths_and_consistencies(result)
segment_data_post = normalize_segment_strengths(segment_data_pre)

In [34]:
segment_data_post

[{'tokens': ['<think>',
   'Ċ',
   'Okay',
   ',',
   'Ġso',
   'ĠI',
   'Ġneed',
   'Ġto',
   'Ġfigure',
   'Ġout',
   'Ġhow',
   'Ġmany',
   'Ġendpoints',
   'ĠFigure',
   'Ġ',
   '5',
   'Ġwill',
   'Ġhave',
   'Ġif',
   'Ġwe',
   'Ġcontinue',
   'Ġthe',
   'Ġpattern',
   'Ġshown',
   '.'],
  'strength': 29.945496113970876,
  'normalized_strength_pre': np.float64(5.9890992227941755),
  'consistency': 0.5151549258042327,
  'normalized_strength_post': np.float64(0.21483106247523465)},
 {'tokens': ['ĠLet',
   'Ġme',
   'Ġstart',
   'Ġby',
   'Ġunderstanding',
   'Ġthe',
   'Ġpattern',
   'Ġfrom',
   'Ġthe',
   'Ġgiven',
   'Ġfigures',
   '.'],
  'strength': 2.4042061576619744,
  'normalized_strength_pre': np.float64(0.6940345361567485),
  'consistency': 0.957031878528003,
  'normalized_strength_post': np.float64(0.02489525907829245)},
 {'tokens': ['ĠĊĊ',
   'First',
   ',',
   'Ġlet',
   'Ġme',
   'Ġlook',
   'Ġat',
   'ĠFigure',
   'Ġ',
   '1',
   '.'],
  'strength': 2.40562432911247,

In [35]:
for i, d in enumerate(segment_data_post):
    d['orig_idx'] = i
    
segments_sorted = sorted(
    segment_data_post,
    key=lambda d: d['normalized_strength_post'],
    reverse=True
)

segments_sorted

[{'tokens': ['<think>',
   'Ċ',
   'Okay',
   ',',
   'Ġso',
   'ĠI',
   'Ġneed',
   'Ġto',
   'Ġfigure',
   'Ġout',
   'Ġhow',
   'Ġmany',
   'Ġendpoints',
   'ĠFigure',
   'Ġ',
   '5',
   'Ġwill',
   'Ġhave',
   'Ġif',
   'Ġwe',
   'Ġcontinue',
   'Ġthe',
   'Ġpattern',
   'Ġshown',
   '.'],
  'strength': 29.945496113970876,
  'normalized_strength_pre': np.float64(5.9890992227941755),
  'consistency': 0.5151549258042327,
  'normalized_strength_post': np.float64(0.21483106247523465),
  'orig_idx': 0},
 {'tokens': ['ĠWait',
   ',',
   'Ġactually',
   ',',
   'Ġlooking',
   'Ġat',
   'Ġthe',
   'ĠAs',
   'ym',
   'pt',
   'ote',
   'Ġcode',
   ',',
   'ĠFigure',
   'Ġ',
   '1',
   'Ġis',
   'Ġdrawn',
   'Ġwith',
   'Ġthree',
   'Ġline',
   'Ġsegments',
   ':',
   'Ġfrom',
   'Ġ(',
   '0',
   ',',
   '0',
   ')',
   'Ġdown',
   'Ġto',
   'Ġ(',
   '0',
   ',-',
   '3',
   '),',
   'Ġthen',
   'Ġfrom',
   'Ġ(',
   '0',
   ',',
   '0',
   ')',
   'Ġto',
   'Ġ(-',
   '2',
   ',',
   '2',
   

In [36]:
TAU = 0.8
BETA = 0.8

k_star = 0
total_normalized_strength = 0
for item in segments_sorted:
    total_normalized_strength += item['normalized_strength_post']
    k_star += 1
    if total_normalized_strength > TAU:
        break
    
print(len(segments_sorted))
top_segments = segments_sorted[:k_star]
print(len(top_segments))

important_segments = []
for item in top_segments:
    if item['consistency'] < BETA:
        important_segments.append(item)
        
print(len(important_segments))

280
84
54


In [37]:
for seg in important_segments:
    print(seg['orig_idx'])

0
279
272
3
17
276
275
58
13
15
43
9
80
273
28
72
278
274
8
6
41
59
76
7
242
95
32
74
18
38
100
227
65
26
12
267
271
269
36
233
91
57
234
257
29
49
258
66
270
56
84
241
236
127


In [38]:
important_segments_orig_order = sorted(
    important_segments,
    key=lambda data: data['orig_idx'],
    reverse=False
)

for seg in important_segments_orig_order:
    print(seg['orig_idx'])

0
3
6
7
8
9
12
13
15
17
18
26
28
29
32
36
38
41
43
49
56
57
58
59
65
66
72
74
76
80
84
91
95
100
127
227
233
234
236
241
242
257
258
267
269
270
271
272
273
274
275
276
278
279


In [40]:
for seg in important_segments_orig_order:
    print(seg['orig_idx']+1, '---', " ".join(tok.replace("Ġ", " ").replace("ĊĊ", "") for tok in seg['tokens']))

1 --- <think> Ċ Okay ,  so  I  need  to  figure  out  how  many  endpoints  Figure   5  will  have  if  we  continue  the  pattern  shown .
4 ---  It 's  a  single  line  segment  with  three  segments  coming  out  of  the  top ,  forming  a  Y  shape .
7 ---  So ,  the  endpoints  here  would  be  the  tips  of  these  three  lines .
8 ---  The  bottom  tip  is  at  ( 0 ,- 3 ),  and  the  two  upper  tips  are  at  (- 2 , 2 )  and  ( 2 , 2 ).
9 ---  So ,  Figure   1  has   3  endpoints .
10 ---   Now ,  moving  to  Figure   2 .
13 ---  The  description  says  each  line -se gment  extrem ity  is  replaced  by  a  gradually  smaller  Y .
14 ---  So ,  in  Figure   1 ,  each  endpoint  is  replaced  by  a  Y  in  Figure   2 .
16 ---   Looking  at  the  As ym pt ote  code  for  Figure   2 :  there 's  a  vertical  line  from  ( 5 , 0 )  to  ( 5 ,- 2 ),  then  a  horizontal  line  connecting  ( 4 ,- 3 )  to  ( 5 ,- 2 )  to  ( 6 ,- 3 ).
18 ---  Then  lines  from  ( 3 , 1 )  to  ( 4 , 1 ) 

In [ ]:
import json
data = []
with open('../data/exp_2_hsp_res.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

In [ ]:
# Steps up to handoff success
DATA['steps'][:DATA['first_essp_index']+1]

['Okay, so I need to figure out how many endpoints Figure 5 will have if we continue the pattern shown.',
 'Let me start by understanding the pattern from the given figures.',
 'First, let me look at Figure 1.',
 "It's a single line segment with three segments coming out of the top, forming a Y shape.",
 'Wait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).',
 "So, it's like a central point with three lines coming out: one straight down, and two going up to the sides.",
 'So, the endpoints here would be the tips of these three lines.',
 'The bottom tip is at (0,-3), and the two upper tips are at (-2,2) and (2,2).',
 'So, Figure 1 has 3 endpoints.',
 'Now, moving to Figure 2.',
 'The Asymptote code draws a more complex figure.',
 'Let me try to visualize it.',
 'The description says each line-segment extremity is replaced by a gradually smaller Y.',
 'So, in Figure 1, each endp

In [ ]:
prompt = DATA['problem']

pruned_cot = ""
for item in important_segments_orig_order:
    if item['orig_idx'] > 73:
        break
    segment_step = DATA['steps'][item['orig_idx']]
    pruned_cot += "\n" + segment_step
    #segment_detokenized = model.tokenizer(segment_tokens, add_special_tokens=False)
    #print(segment_detokenized)
    #for id in segment_detokenized['input_ids']:
    #    assert len(id) == 1, print(id, model.tokenizer.decode(id))
    #segment_textified = model.tokenizer.decode()

pruned_cot += "\n</think>\nTherefore, the final answer is \\boxed{"

prompt += pruned_cot

print(prompt)


If you continue this pattern in which each line-segment extremity is replaced by a gradually smaller Y in the next figure, in the manner shown, how many endpoints will Figure 5 have?

[asy]
draw((0,0)--(0,-3),linewidth(.75));
draw((0,0)--(-2,2),linewidth(.75));
draw((0,0)--(2,2),linewidth(.75));
label("Figure 1",(0,-3),S);

draw((5,0)--(5,-2),linewidth(.75));
draw((4,-3)--(5,-2)--(6,-3),linewidth(.75));
draw((4,1)--(5,0)--(6,1),linewidth(.75));
draw((3,1)--(4,1)--(4,2),linewidth(.75));
draw((6,2)--(6,1)--(7,1),linewidth(.75));
label("Figure 2",(5,-3),S);

draw((10,0)--(10,-2),linewidth(.75));
draw((9.5,-2.5)--(10,-2)--(10.5,-2.5),linewidth(.75));
draw((9,-2.5)--(9.5,-2.5)--(9.5,-3),linewidth(.75));
draw((11,-2.5)--(10.5,-2.5)--(10.5,-3),linewidth(.75));

draw((9,1)--(10,0)--(11,1),linewidth(.75));
draw((8.5,1)--(9,1)--(9,1.5),linewidth(.75));
draw((11.5,1)--(11,1)--(11,1.5),linewidth(.75));
draw((8.25,.75)--(8.5,1)--(8.25,1.25),linewidth(.75));
draw((8.75,1.75)--(9,1.5)--(9.25,1.75),li

In [55]:
tokenized_cot = model.tokenizer(prompt, return_tensors='pt')
outs = []
for i in range(20):
    with model.generate(tokenized_cot, max_new_tokens=2, temperature=0.7):
        out = model.generator.output.save()
    outs.append(model.tokenizer.decode(out[0][-3:]))
    

In [53]:
outs

['boxed{4', 'boxed{4', 'boxed{4', 'boxed{9', 'boxed{4']